# 01 — Adatbetöltés, tisztítás és feltáró adatelemzés (EDA)

**Szakdolgozat:** Sentiment Analysis of Customer Reviews using Machine Learning Techniques
**Témavezető:** Mutaz AlShafeey
**Adat:** a Booking.com adathalmaz `val.csv` fájlja (~200 000 review)

---

## Mit csinálunk ebben a notebookban?

| # | Lépés | Dolgozat-fejezet |
|---|---|---|
| 1 | Környezet és importok | — |
| 2 | Adat betöltése + **a névütközés tisztázása** | 4. Data Sources |
| 3 | Adatminőség: hiányzó értékek, duplikátumok, hosszak | 6.1 Preprocessing |
| 4 | Sentiment-címkék + **küszöb-érzékenységi vizsgálat** | 6.1 Preprocessing |
| 5 | Szűrés és tisztítás | 6.1 Preprocessing |
| 6 | Alapvető EDA-ábrák | 9. EDA |
| 7 | **Leíró statisztikák és kereszttáblák** | 9. EDA |
| 8 | Aspektus-előzetes | 7. ABSA / 9. EDA |
| 9 | 60/20/20 felosztás és mentés | 8. Data Split |

> **Minden ábra és táblázat automatikusan mentődik** a `results/` mappába.
> Amikor a dolgozatot írod, onnan másolsz — nem futtatod újra ezt a notebookot.

## 1. Környezet és importok

In [ ]:
# A notebooks/ mappából futunk, ezért a szülőmappát hozzá kell adni az
# elérési úthoz, hogy a `src` csomagot importálni tudjuk.
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src import set_seed
from src.config import CFG, PATHS, SENTIMENT_COLORS, INK, apply_plot_style
from src import data_loading as dl
from src import preprocessing as pp
from src import descriptives as ds
from src import aspects as asp
from src import evaluation as ev

# Reprodukálhatóság: EZ A LEGFONTOSABB SOR a notebookban.
# Enélkül minden futtatás kicsit más eredményt ad.
set_seed()
PATHS.ensure()

# Egységes ábrastílus az egész dolgozathoz (színek, betűméret, 300 DPI mentés)
apply_plot_style()

pd.set_option("display.max_colwidth", 120)
pd.set_option("display.width", 170)

print(f"pandas {pd.__version__} | numpy {np.__version__}")
print(f"Seed: {CFG.random_seed}")
print(f"Adatfájl: data/raw/{CFG.local_csv_name}")

## 2. Adat betöltése

### ⚠ Fontos névtisztázás — ezt a dolgozatban is le kell írni

A fájl neve `val.csv`, mert ez a **Booking.com saját** train/validation/test
felosztásának validációs része. **Nálunk viszont ez a teljes munkaadathalmaz**,
amit MI osztunk fel 60/20/20 arányban saját tanító/validációs/teszt halmazra.

```
Booking.com felosztása:     train.csv    val.csv ←EZT HASZNÁLJUK    test.csv
                                            │
                            ┌───────────────┴───────────────┐
A MI felosztásunk:       train (60%)   validation (20%)   test (20%)
```

A két "validation" szónak **semmi köze egymáshoz**. A kódban ezért hívjuk a
betöltött adathalmazt `booking_val_split`-nek, nem `validation`-nek.

**Amit a dolgozat 4. fejezetébe írj:** „A kísérletekhez a Booking.com
adathalmaz publikált validációs részhalmazát használtam (kb. 200 000 review).
Ez a teljes adathalmaz véletlenszerű részhalmaza, így reprezentatívnak
tekinthető; a számításigény csökkentése érdekében esett rá a választás."

> Ez egy teljesen legitim döntés — 200 ezer review bőven elegendő ehhez a
> feladathoz. De **indokolni kell**, különben a bíráló megkérdezi, miért nem a
> teljes adatot használtad.

In [ ]:
# A loader felismeri az oszlopneveket, és kiírja, mit talált.
# Ha valamelyik oszlop hiányzik, figyelmeztet — nézd meg a kimenetet!
df_raw = dl.load_dataset_by_name("booking_local")

print(f"\nBetöltve: {len(df_raw):,} sor, {df_raw.shape[1]} oszlop")
print(f"Oszlopok: {list(df_raw.columns)}")

In [ ]:
# Nézz meg néhány valódi sort — MINDIG, minden adathalmaznál.
# Sok hibát csak így veszel észre (helykitöltő szövegek, kódolási hibák).
df_raw.head(3)

> **Ha a fenti cella hibát dobott** („Nem találom a pontszám oszlopot"), akkor a
> CSV oszlopnevei eltérnek a várttól. A hibaüzenet kiírja a tényleges neveket —
> küldd át, és kiegészítem a `_BOOKING_ALIASES` szótárat a `data_loading.py`-ban.

## 3. Adatminőség

In [ ]:
print("=== Típusok ===")
print(df_raw.dtypes.to_string())

print("\n=== Hiányzó értékek ===")
miss = ds.missing_report(df_raw)
print(miss.to_string() if len(miss) else "Nincs hiányzó érték.")

print("\n=== Duplikátumok (szöveg alapján) ===")
n_dup = df_raw["text"].duplicated().sum()
print(f"{n_dup:,} db ({n_dup/len(df_raw):.2%})")

print("\n=== Pontszám ===")
print(df_raw["score"].describe().to_string())

In [ ]:
# Szöveghossz — ez dönti el, mekkora max_length kell majd a BERT-nek.
df_raw["n_chars"] = df_raw["text"].str.len()
df_raw["n_words"] = df_raw["text"].str.split().str.len()

print(df_raw[["n_chars", "n_words"]].describe(
    percentiles=[.5, .75, .9, .95, .99]).round(1).to_string())

q95 = df_raw["n_words"].quantile(0.95)
print(f"\nA reviewk 95%-a rövidebb, mint {q95:.0f} szó.")
print(f"A CFG.max_length = {CFG.max_length} token ebből kb. "
      f"{'ELÉG' if CFG.max_length > q95 * 1.3 else 'KEVÉS LEHET — fontold meg a 256-ot'}.")
print("(Ökölszabály: 1 szó ≈ 1,3 token angolban.)")

In [ ]:
# Konkrét példák három pontszám-tartományból.
# Figyeld meg, mennyire NEM különül el nyelvileg a középső sáv!
for lo, hi, cim in [(1, 5.9, "ALACSONY"), (6.0, 7.9, "KÖZEPES"), (8.0, 10, "MAGAS")]:
    sub = df_raw[(df_raw["score"] >= lo) & (df_raw["score"] <= hi)]
    print(f"\n{'='*70}\n{cim} pontszám ({lo}–{hi}) — {len(sub):,} review\n{'='*70}")
    for _, r in sub.sample(min(2, len(sub)), random_state=CFG.random_seed).iterrows():
        print(f"[{r['score']}] {r['text'][:220]}")

## 4. Sentiment-címkék képzése

**Ez a dolgozat egyik legfontosabb módszertani döntése.**

Az adatban nincs kész `positive / neutral / negative` címke, csak egy 1–10-es
vendégpontszám. Ebből képezzük a három osztályt:

| Feltétel | Címke |
|---|---|
| `score ≤ 6` | negative |
| `6 < score < 8` | neutral |
| `score ≥ 8` | positive |

A 6/8 vágás a szakirodalomban gyakori, de **önkényes**. Ezért futtatunk alább egy
**érzékenységi vizsgálatot**: megnézzük, mennyire változik az osztályeloszlás más
küszöbökkel. Ez pont az a módszertani alaposság, amit a bírálók keresnek.

In [ ]:
df = pp.add_sentiment_labels(df_raw)

dist = df["sentiment"].value_counts().reindex(list(CFG.labels))
summary = pd.DataFrame({"db": dist, "arány": (dist / len(df)).round(4)})
display(summary)

imbalance = dist.max() / dist.min()
print(f"Kiegyensúlyozatlansági arány: {imbalance:.1f}x")
print("=> Ezért lesz a MACRO-F1 a fő metrikánk (nem az accuracy),")
print("   és ezért használunk class_weight='balanced'-et a modelleknél.")

In [ ]:
# ÉRZÉKENYSÉGI VIZSGÁLAT — ez a táblázat megy a dolgozat 6.1 fejezetébe.
rows = []
for neg_t, pos_t in [(5.0, 8.0), (6.0, 8.0), (6.0, 8.5), (6.5, 8.5), (7.0, 9.0)]:
    labels = [pp.score_to_sentiment(s, sm, neg_threshold=neg_t, pos_threshold=pos_t)
              for s, sm in zip(df_raw["score"], df_raw["score_scale_max"])]
    vc = pd.Series(labels).value_counts(normalize=True)
    rows.append({
        "küszöbök": f"neg≤{neg_t} / pos≥{pos_t}",
        "negative": vc.get("negative", 0),
        "neutral":  vc.get("neutral", 0),
        "positive": vc.get("positive", 0),
        "kiegyensúlyozatlanság": vc.max() / vc.min(),
    })

sens = pd.DataFrame(rows).set_index("küszöbök").round(3)
display(sens)
ev.save_table(sens, "01_label_threshold_sensitivity")

print("\nMit olvass ki belőle:")
print("  • A neutral osztály mérete ERŐSEN függ a küszöbtől.")
print("  • Ha a jelenlegi beállítással a neutral 10% alatt van, érdemes")
print("    szélesíteni a sávot — különben a modellnek túl kevés példája lesz.")

## 5. Szűrés és tisztítás

In [ ]:
# A build_dataset() elvégzi: szűrés -> címkézés -> kétféle tisztítás.
#
# A kétféle tisztítás SZÁNDÉKOS:
#   text_clean  -> TF-IDF-hez: agresszív (kisbetű, írásjelek ki, stopword ki,
#                  lemmatizálás). A zsák-a-szavak modellnek segít a zajszűrés.
#   text_bert   -> BERT-hez: minimális. A transformer a kontextusból tanul;
#                  ha kidobnád a tagadást, a "not clean" -> "clean" lenne,
#                  vagyis az ELLENKEZŐJÉT tanítanád neki.
#
# 200k soron ez néhány percig fut (a lemmatizálás a lassú rész).
df = pp.build_dataset(df_raw, classic_clean=True, minimal_clean=True, balance="none")

df["n_words"] = df["text"].str.split().str.len()
df["n_chars"] = df["text"].str.len()

print(f"\nVégleges méret: {len(df):,} sor")
print(df["sentiment"].value_counts().to_string())

In [ ]:
# Mit CSINÁLT a tisztítás? Ezt a példát tedd be a dolgozat 6.1 fejezetébe!
ex = df.iloc[0]
print("EREDETI:\n ", ex["text"][:280])
print("\nBERT-hez (minimális tisztítás):\n ", ex["text_bert"][:280])
print("\nTF-IDF-hez (teljes tisztítás):\n ", ex["text_clean"][:280])

## 6. Alapvető EDA-ábrák

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4.2))

# (a) Pontszám-eloszlás a küszöbökkel
axes[0].hist(df["score"], bins=40, color="#2a78d6", edgecolor="white", linewidth=0.5)
axes[0].axvline(CFG.neg_threshold, color=SENTIMENT_COLORS["negative"], ls="--", lw=2,
                label=f"neg ≤ {CFG.neg_threshold}")
axes[0].axvline(CFG.pos_threshold, color=SENTIMENT_COLORS["positive"], ls="--", lw=2,
                label=f"pos ≥ {CFG.pos_threshold}")
axes[0].set(xlabel="Vendégpontszám", ylabel="Reviewk száma", title="(a) Pontszám-eloszlás")
axes[0].legend(fontsize=9)

# (b) Osztályeloszlás
order = list(CFG.labels)
counts = df["sentiment"].value_counts().reindex(order)
axes[1].bar(order, counts, color=[SENTIMENT_COLORS[l] for l in order], width=0.68)
for i, v in enumerate(counts):
    axes[1].text(i, v, f"{v:,}\n({v/len(df):.1%})", ha="center", va="bottom", fontsize=9)
axes[1].set(ylabel="Reviewk száma", title="(b) Sentiment-osztályok")
axes[1].margins(y=0.20)
axes[1].grid(axis="x", visible=False)

# (c) Szöveghossz osztályonként
data = [df.loc[df["sentiment"].astype(str) == l, "n_words"] for l in order]
bp = axes[2].boxplot(data, labels=order, patch_artist=True, showfliers=False,
                     medianprops=dict(color=INK["primary"], linewidth=1.6),
                     widths=0.55)
for patch, l in zip(bp["boxes"], order):
    patch.set_facecolor(SENTIMENT_COLORS[l]); patch.set_edgecolor("white")
axes[2].set(ylabel="Szavak száma", title="(c) Reviewhossz osztályonként")
axes[2].grid(axis="x", visible=False)

plt.tight_layout()
ev.save_figure("01_eda_overview")
plt.show()

In [ ]:
# Számszerűsítsd, amit a (c) ábrán látsz — ez megy a szövegbe.
lens = df.groupby("sentiment", observed=True)["n_words"].agg(["count", "mean", "median"]).round(1)
display(lens)

neg_m = lens.loc["negative", "median"]; pos_m = lens.loc["positive", "median"]
print(f"A negatív reviewk mediánhossza {neg_m:.0f} szó, a pozitívaké {pos_m:.0f}.")
print("\nTipikus megfigyelés: a NEGATÍV reviewk hosszabbak — az elégedetlen vendég")
print("részletesen leírja a problémát, míg a pozitív értékelés gyakran pár szó.")
print("Ez önmagában is prediktív jel, és a hibaelemzésnél vissza fog térni.")

In [ ]:
# Leggyakoribb kifejezések osztályonként — az első betekintés abba,
# MI hajtja a sentimentet.
from sklearn.feature_extraction.text import CountVectorizer

fig, axes = plt.subplots(1, 3, figsize=(15, 4.8))
for ax, label in zip(axes, CFG.labels):
    texts = df.loc[df["sentiment"].astype(str) == label, "text_clean"]
    if len(texts) == 0:
        continue
    cv = CountVectorizer(max_features=15, ngram_range=(1, 2), min_df=3)
    X = cv.fit_transform(texts)
    freq = pd.Series(np.asarray(X.sum(axis=0)).ravel(),
                     index=cv.get_feature_names_out()).sort_values()
    ax.barh(freq.index, freq.values, color=SENTIMENT_COLORS[label], height=0.72)
    ax.set_title(f"{label}  (n={len(texts):,})", loc="left")
    ax.set_xlabel("Előfordulás")
    ax.grid(axis="y", visible=False)

plt.tight_layout()
ev.save_figure("01_top_words_by_sentiment")
plt.show()

## 7. Leíró statisztikák és kereszttáblák

Itt lépünk túl azon, hogy "hány sor van az adatban". A Booking.com metaadatai
(ország, szállástípus, vendégtípus, csillagbesorolás) lehetővé teszik, hogy a
sentimentet **ezek mentén** vizsgáld — ezt csinálná egy szállodalánc elemzője is.

### A központi mutató: a LIFT

Nem elég azt mondani, hogy „a hostelek 35%-a negatív". Tudni kell, hogy ez sok-e:

```
lift = a csoport negatív aránya / a teljes adathalmaz negatív aránya

lift = 1,0  →  pontosan átlagos
lift = 1,4  →  40%-kal gyakrabban negatív az átlagnál
lift = 0,7  →  30%-kal ritkábban
```

Így az eltérő méretű csoportok is összehasonlíthatók, és egyetlen szám megmondja,
hol van a probléma.

### Egy szó a színekről

Az ábrák **piros–szürke–kék** skálát használnak, nem piros–zöldet. A vörös-zöld
színtévesztés a férfiak kb. 8%-át érinti, és pont ezt a két színt mossa össze —
egy kinyomtatott szakdolgozatnál ez valós olvashatósági probléma. A piros–kék pár
színtévesztéssel is elkülöníthető marad.

In [ ]:
# Áttekintő tábla — ez megy a dolgozat 4. és 9. fejezetébe
overview = ds.dataset_overview(df)
display(overview)
ev.save_table(overview, "01_dataset_overview")

In [ ]:
# Numerikus oszlopok profilja
prof = ds.numeric_profile(df)
display(prof)
ev.save_table(prof, "01_numeric_profile")

In [ ]:
# Melyik kategóriából van a legtöbb review? (Ezt tudni kell, mielőtt
# bármilyen csoportos összehasonlítást értelmeznél.)
for col in ["accommodation_country", "accommodation_type", "guest_type"]:
    if col in df.columns:
        ds.plot_top_categories(df, col, top_n=12,
                               title=f"Reviewk megoszlása — {col}",
                               save_as=f"01_counts_{col}")
        plt.show()

In [ ]:
# === A FŐ KERESZTTÁBLA-FUTTATÁS ===
# Minden elérhető dimenzióra: tábla + ábra, automatikusan mentve.
#
# min_count=200: ennél kevesebb reviewval rendelkező kategóriát eldobunk.
#   MIÉRT? Mert 12 reviewból számolt 50%-os negatív arány statisztikai zaj,
#   nem megállapítás. Ezt a küszöböt a dolgozatban meg kell említeni.
crosstabs = ds.run_all_crosstabs(df, min_count=200, top_n=12, save=True)

print(f"\nElkészült {len(crosstabs)} kereszttábla.")

In [ ]:
# Nézzük meg részletesen a szállástípust — jellemzően itt a legnagyobb a szórás
if "accommodation_type" in crosstabs:
    display(crosstabs["accommodation_type"])
    print("\nOlvasat:")
    print("  neg_lift > 1  ->  ez a szállástípus PANASZHAJTÓ")
    print("  neg_lift < 1  ->  ELÉGEDETTSÉGHAJTÓ")

In [ ]:
# Az ábrák már elmentődtek; itt újra megjelenítjük őket a notebookban
for col, tab in crosstabs.items():
    ds.plot_sentiment_by(tab, f"Sentiment-megoszlás — {col}")
    plt.show()

### Két dimenzió együtt — hőtérkép

Az egydimenziós táblák elrejthetnek egy összefüggést. Például lehet, hogy a
hostelek azért tűnnek rosszabbnak, mert jellemzően alacsonyabb csillagbesorolásúak
— nem azért, mert hostelek. A kereszttábla ezt szétválasztja.

A hőtérkép **divergáló** skálát használ: kék = az átlagnál **kevesebb** negatív,
piros = **több**. Azért divergálót, mert a kérdés nem „mennyi?", hanem „az
átlaghoz képest jobb vagy rosszabb?" — ennek van értelmes közepe (lift = 1,0).

In [ ]:
def heatmap(rows, cols, xlabel, ylabel, fname, min_count=100, top_rows=8, top_cols=6):
    """Hőtérkép egy sorban, hibatűrően.

    Ha minden cella kimaszkolódik (túl kevés review), nem áll meg a notebook,
    hanem kiírja, mit állíts át. Kis mintán ez elő szokott fordulni."""
    if not {rows, cols} <= set(df.columns):
        print(f"Kihagyva — nincs meg mindkét oszlop: {rows}, {cols}")
        return
    try:
        lift, counts, base = ds.negativity_heatmap(
            df, rows=rows, cols=cols,
            min_count=min_count, top_rows=top_rows, top_cols=top_cols,
        )
    except ValueError as e:
        print(f"Kihagyva: {e}")
        return
    display(lift)
    ev.save_table(lift, fname)
    ds.plot_negativity_heatmap(
        lift, counts, base,
        title="Negatív értékelések aránya az átlaghoz képest",
        xlabel=xlabel, ylabel=ylabel, save_as=fname,
    )
    plt.show()
    print(f"A szürke cellák: kevesebb mint {min_count} review — nem értelmezhető.")


# Szállástípus × csillagbesorolás
heatmap("accommodation_type", "accommodation_star_rating",
        xlabel="Csillagbesorolás", ylabel="Szállástípus",
        fname="01_heatmap_type_x_stars")

In [ ]:
# Ország × szállástípus
heatmap("accommodation_country", "accommodation_type",
        xlabel="Szállástípus", ylabel="Ország",
        fname="01_heatmap_country_x_type", top_rows=10)

> **Írd le a naplóba, amit LÁTSZ** (`docs/naplo.md`):
> - Melyik szállástípus/ország a leginkább panaszhajtó, és mennyivel?
> - Eltűnik-e a hatás, ha a csillagbesorolást is figyelembe veszed? (Ha igen,
>   akkor a csillagbesorolás volt az igazi magyarázó változó — ez fontos
>   megkülönböztetés, és jól mutatja, miért kell kereszttábla.)
> - Van-e olyan vendégtípus, amelyik szisztematikusan szigorúbban értékel?
>
> **Figyelmeztetés a megfogalmazásra:** ezek **asszociációk**, nem oksági
> viszonyok. Ne azt írd, hogy „a hostel rosszabb élményt ad", hanem hogy
> „a hostel kategóriában az átlagnál gyakoribb a negatív értékelés". A dolgozatod
> 7. fejezete már helyesen rögzíti ezt a különbséget.

## 8. Aspektus-előzetes

In [ ]:
# Mondatra bontás + kulcsszavas aspektusdetektálás.
# Ez a BASELINE, amihez a későbbi transformer-alapú ABSA-t hasonlítod (RQ4).
# Lassú művelet, ezért mintán futtatjuk.
sample = df.sample(min(30_000, len(df)), random_state=CFG.random_seed)

sent_df = asp.explode_to_sentences(sample, text_col="text")
sent_df = asp.add_aspect_columns(sent_df, text_col="sentence")

print(f"{len(sample):,} review -> {len(sent_df):,} mondat")
print(f"Átlagosan {sent_df['n_aspects'].mean():.2f} aspektus / mondat")
print(f"A mondatok {(sent_df['n_aspects'] == 0).mean():.1%}-ában nem talált aspektust.")
print("\n=> Ezek az IMPLICIT aspektusok ('we could not sleep' = zaj, kulcsszó")
print("   nélkül). A transformer ezek egy részét meg fogja találni — ez lesz")
print("   az egyik fő érved amellett, miért kell transformer-alapú ABSA.")

In [ ]:
# RQ3 ELSŐ VÁLASZA: mely aspektusok kötődnek a negatív sentimenthez?
aspect_table = asp.aspect_sentiment_table(sent_df, min_count=100)
display(aspect_table)
ev.save_table(aspect_table, "01_aspect_sentiment_distribution")

asp.plot_aspect_sentiment(aspect_table, save_as="01_aspect_sentiment_stacked")
plt.show()

In [ ]:
# Konkrét példák: hogyan néz ki a "textuális bizonyíték" kiemelése?
neg_sent = sent_df[(sent_df["sentiment"].astype(str) == "negative") &
                   (sent_df["n_aspects"] > 0)].head(8)

for _, r in neg_sent.iterrows():
    print(f"MONDAT: {r['sentence'][:150]}")
    for a in r["aspects"]:
        spans = asp.extract_evidence(r["sentence"], a)
        if spans:
            flag = "   [TAGADÁS!]" if asp.has_negation(spans[0]) else ""
            print(f'   • {a:16s} -> "{spans[0]}"{flag}')
    print()

> **Gyűjtsd a naplóba, hol bukik el a kulcsszavas módszer.** Keress példákat:
> tagadás („the room was **not** clean"), irónia, implicit aspektus, több
> aspektus egy mondatban ellentétes sentimenttel. Ez a lista lesz az RQ4
> indoklása — és egy önálló, értékes alfejezet.
>
> **Módszertani korlát, amit le kell írni:** itt a *review-szintű* sentimentet
> vetítjük a mondatokra, ami közelítés. Egy pozitív összpontszámú review is
> tartalmazhat negatív mondatot. A pontos mondat-szintű címkékhez kell a HRAST
> adathalmaz (4. fázis).

## 9. Felosztás és mentés

In [ ]:
train_df, val_df, test_df = pp.train_val_test_split(df)

# Ellenőrzés: megmaradt az osztályarány mindhárom halmazban?
check = pd.DataFrame({
    "train": train_df["sentiment"].value_counts(normalize=True),
    "val":   val_df["sentiment"].value_counts(normalize=True),
    "test":  test_df["sentiment"].value_counts(normalize=True),
}).reindex(list(CFG.labels)).round(4)
display(check)
print("A három oszlopnak szinte azonosnak kell lennie -> a rétegzés működik.")

In [ ]:
# Mentés. A 02-es és 03-as notebook INNEN tölt be, így mindhárom modell
# PONTOSAN ugyanazon a felosztáson dolgozik — enélkül az RQ1 összehasonlítás
# értelmetlen lenne.
keep = ["review_id", "text", "text_clean", "text_bert", "score", "sentiment", "dataset"]
# A metaadatokat is visszük, hogy a hibaelemzésnél csoportosítani lehessen
# (pl. "melyik szállástípusnál hibázik legtöbbet a modell?")
keep += [c for c in ["accommodation_type", "accommodation_country",
                     "accommodation_star_rating", "guest_type"] if c in df.columns]

dl.save_splits({
    "train": train_df[keep],
    "val":   val_df[keep],
    "test":  test_df[keep],
})

# A konfigurációt is mentsük az adat mellé — így minden későbbi eredményhez
# tudni fogod, pontosan milyen beállításokkal készült ez a felosztás.
CFG.to_json(PATHS.processed / "config_snapshot.json")
print("\nKonfiguráció elmentve: config_snapshot.json")

---

## Összefoglaló — mi készült el

### Ábrák (`results/figures/`)

| Fájl | Dolgozat-fejezet |
|---|---|
| `01_eda_overview.png` | 9. EDA — pontszám, osztályok, hosszak |
| `01_top_words_by_sentiment.png` | 9. EDA — mi hajtja a sentimentet |
| `01_counts_*.png` | 9. EDA — a minta összetétele |
| `01_crosstab_*.png` | **9. EDA — a leíró elemzés fő ábrái** |
| `01_heatmap_*.png` | 9. EDA — két dimenzió együtt |
| `01_aspect_sentiment_stacked.png` | 7. ABSA / RQ3 |

### Táblázatok (`results/tables/`)

| Fájl | Dolgozat-fejezet |
|---|---|
| `01_dataset_overview.csv` | 4. Data Sources |
| `01_label_threshold_sensitivity.csv` | **6.1 — a küszöbválasztás indoklása** |
| `01_numeric_profile.csv` | 9. EDA |
| `01_crosstab_*.csv` | 9. EDA |
| `01_heatmap_*.csv` | 9. EDA |
| `01_aspect_sentiment_distribution.csv` | **RQ3 első válasza** |

### Adat (`data/processed/`)

`booking_train.parquet`, `booking_val.parquet`, `booking_test.parquet` —
ezekből dolgozik a 02-es és a 03-as notebook.

---

## MOST írd meg a naplót

Nyisd meg a `docs/naplo.md`-t, és válaszolj:

1. **Miért a `val.csv`?** (részhalmaz, számításigény, reprezentativitás)
2. **Mi a licenc, és hogyan hivatkozod?** (CC BY-SA 4.0)
3. **Miért 6/8 a küszöb?** — az érzékenységi táblázattal alátámasztva
4. **Mit dobott ki a szűrés?** — a szűrési napló konkrét számaival
5. **Mi a feltűnő az eloszlásokban?** (kiegyensúlyozatlanság, hosszkülönbség)
6. **Mit mutatnak a kereszttáblák?** — melyik csoport panaszhajtó, és eltűnik-e
   a hatás, ha egy másik dimenziót is bevonsz
7. **Hol bukik el a kulcsszavas aspektusdetektálás?**

> Ez a hét pont a dolgozat 4., 6.1., 7. és 9. fejezetének a nyersanyaga.
> Sokkal könnyebb most leírni, mint három hónap múlva visszaemlékezni rá.

**Commit:**
```
git add .
git commit -m "01-es notebook lefutott: EDA, kereszttablak, split elmentve"
```

**Következő:** `02_classical_baseline.ipynb` — TF-IDF + Naive Bayes / LogReg / SVM.